# Part 12 — Steering it: guidance, latents, and Stable Diffusion

_Rigorous Courses · Diffusion Models — Part 12 of 12_

**Add the steering wheel with one Bayes tilt on the score, shrink the canvas with latents, and assemble Stable Diffusion from parts you built yourself**

In this notebook you finish the course. First you train a conditional DDPM with label dropout and steer it with classifier-free guidance at $w = 0, 1, 3, 7$ — watching the knob you derived by hand move samples onto a chosen cluster. Then the finale: latent diffusion in miniature. PCA compresses the digits from 64 pixels to 16 latent numbers, a small DDPM learns the latent cloud, and the decoded samples are recognizable digit images generated from pure noise on a CPU.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab; torch is preinstalled there too.
import math

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
torch.manual_seed(0)

print(f"torch version: {torch.__version__} (CPU is all we need)")

## Conditional data: the eight-mode ring, now with labels

Same dataset as parts 10 and 11 — 4,096 points in 8 Gaussian clusters around a circle of radius 4 — but this time we **keep the cluster index as a label** $y \in \{0, \dots, 7\}$. Conditional generation will mean: "give me a sample from cluster 2", a draw from $p(x \mid y = 2)$ rather than from the whole ring.

### Step 1 — Build the labeled ring

The construction is part 10's, with one change: the cluster index each point is born from is no longer thrown away — it becomes the label. We check the shapes and that every label appears roughly equally often (the lottery each cluster wins about 1/8 of the time).

In [ ]:
n = 4096

labels_np = rng.integers(0, 8, size=n)
angle = 2 * np.pi * labels_np / 8
centers = 4.0 * np.stack([np.cos(angle), np.sin(angle)], axis=1)
x0_np = centers + 0.15 * rng.standard_normal((n, 2))

data = torch.tensor(x0_np, dtype=torch.float32)
labels = torch.tensor(labels_np, dtype=torch.long)
label_counts = np.bincount(labels_np, minlength=8)

assert data.shape == (4096, 2)
assert labels.shape == (4096,)
assert int(labels.min()) == 0
assert int(labels.max()) == 7

print(f"data shape: {tuple(data.shape)}   labels shape: {tuple(labels.shape)}")
print(f"points per label: {label_counts.tolist()}")
print(f"label shares range from {label_counts.min() / n:.3f} to {label_counts.max() / n:.3f} (the 1/8 lottery = 0.125)")

assert label_counts.min() > n / 16

### Step 2 — Look at the labeled cloud

Same eight blobs, now colored by label. Conditioning on $y = 2$ means restricting the draw to one colored blob — part 2's conditional distribution, made visible.

In [ ]:
plt.figure(figsize=(5.5, 5))
sc = plt.scatter(x0_np[:, 0], x0_np[:, 1], s=4, alpha=0.5, c=labels_np, cmap="tab10")
plt.colorbar(sc, label="cluster label y")
plt.title("Eight-mode ring with labels: p(x | y) is one colored blob")
plt.xlabel("first coordinate")
plt.ylabel("second coordinate")
plt.axis("equal")
plt.show()

## The schedule, and the formula this part owns

The noise schedule is untouched from parts 6 and 10: $T = 200$, linear $\beta_t$ from $10^{-4}$ to $0.02$. What is new is what we will do with the network's guesses at sampling time. With $\epsilon_y = \epsilon_\theta(x_t, t, y)$ (the labeled call) and $\epsilon_\varnothing = \epsilon_\theta(x_t, t, \varnothing)$ (the null-token call), **classifier-free guidance** hands the sampler

$$\tilde\epsilon \;=\; \epsilon_\varnothing \;+\; w\,\big(\epsilon_y - \epsilon_\varnothing\big)$$

— the unconditional guess plus $w$ times the steering direction. The lesson derived it as the score of the sharpened density $q(x_t)\, q(y \mid x_t)^w$, converted to noise-guess units by part 11's bridge.

### Step 3 — Rebuild betas, alphas, abar

Part 10's schedule code, verbatim. We precompute the two square-root lookups the loops use thousands of times.

In [ ]:
T = 200

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)

sqrt_abar = torch.sqrt(abar)
sqrt_1m_abar = torch.sqrt(1.0 - abar)

assert torch.all(abar[1:] < abar[:-1])
assert abar.min() > 0.0
assert abar.max() < 1.0

print(f"beta_1 = {betas[0].item():.6f}   beta_T = {betas[-1].item():.6f}")
print(f"abar_T = {abar[-1].item():.4f} — only 13% of signal variance survives; enough for this toy,")
print("            though the real T=1000 schedule drives it to ~4e-5 (part 6)")

### Step 4 — Check the lesson's CFG arithmetic

The worked example: $\epsilon_\varnothing = (0.2, -0.1)$, $\epsilon_y = (0.8, 0.3)$, $w = 3$. By hand: the steering direction is $(0.6, 0.4)$, scaled to $(1.8, 1.2)$, so $\tilde\epsilon = (2.0, 1.1)$. We also check the two special cases you should know cold: $w = 0$ returns the unconditional guess, $w = 1$ returns the conditional one.

In [ ]:
eps_null = torch.tensor([0.2, -0.1])
eps_lab = torch.tensor([0.8, 0.3])
direction = eps_lab - eps_null
eps_tilde = eps_null + 3.0 * direction

assert torch.allclose(direction, torch.tensor([0.6, 0.4]))
assert torch.allclose(eps_tilde, torch.tensor([2.0, 1.1]))
assert torch.allclose(eps_null + 0.0 * direction, eps_null)
assert torch.allclose(eps_null + 1.0 * direction, eps_lab)

print(f"steering direction eps_y - eps_null: {direction.tolist()}")
print(f"guided guess at w = 3: {eps_tilde.tolist()} (hand answer: [2.0, 1.1])")
print("w = 0 reproduces the unconditional guess; w = 1 reproduces the conditional guess")

## The conditional network: a label embedding, the same trick as t

The network becomes $\epsilon_\theta(x_t, t, y)$. The timestep enters through part 10's sinusoidal embedding, unchanged. The label enters through a **learned embedding table**: 9 rows of 32 numbers — one row per cluster, plus row 8 for the **null token** $\varnothing$ ("no label given") that classifier-free guidance needs. Input: 2 coordinates + 32 time numbers + 32 label numbers = 66. Output: 2 numbers — still data-shaped in, data-shaped out.

### Step 5 — Define the conditional MLP and count its knobs

Hand count: label table $9 \times 32 = 288$; layer 1 is $66 \times 128 + 128 = 8576$; layer 2 is $128 \times 128 + 128 = 16512$; layer 3 is $128 \times 2 + 2 = 258$. Total: $25{,}634$. We assert the code agrees and that a 256-point batch produces a $(256, 2)$ guess.

In [ ]:
emb_dim = 32
half = emb_dim // 2
NULL_LABEL = 8

freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / (half - 1))


def time_embedding(t):
    args = t[:, None].float() * freqs[None, :]
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    return emb


class CondEpsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(9, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(2 + emb_dim + emb_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x, t, y):
        temb = time_embedding(t)
        yemb = self.label_emb(y)
        h = torch.cat([x, temb, yemb], dim=1)
        return self.net(h)


model = CondEpsMLP()
n_params = sum(p.numel() for p in model.parameters())
test_out = model(data[:256], torch.randint(1, T + 1, (256,)), labels[:256])

assert n_params == 25634
assert test_out.shape == (256, 2)

print(f"parameter count: {n_params} (hand count: 288 + 8576 + 16512 + 258 = 25634)")
print(f"output shape for a 256-point batch: {tuple(test_out.shape)} — same shape as the data")

### Step 6 — Train with 10% label dropout

Algorithm 1, with one new line: before each batch's forward pass, 10% of the labels are replaced by the null token. One set of weights learns both jobs — the conditional guess (labeled examples) and the unconditional guess (null-token examples). Everything else is part 10's loop verbatim: loss near 1.0 at the start, noisy fall, floor above zero.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for step in range(2500):
    idx = torch.randint(0, n, (256,))
    x0 = data[idx]
    y = labels[idx].clone()
    drop = torch.rand(256) < 0.1
    y[drop] = NULL_LABEL
    t = torch.randint(1, T + 1, (256,))
    eps = torch.randn(256, 2)
    xt = sqrt_abar[t - 1][:, None] * x0 + sqrt_1m_abar[t - 1][:, None] * eps
    eps_pred = model(xt, t, y)
    loss = ((eps - eps_pred) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 250 == 0 or step == 2499:
        print(f"step {step:4d}   batch loss = {loss.item():.3f}")

early = losses[0]
late = float(np.mean(losses[-100:]))

assert early > 0.7
assert late < early * 0.5

print(f"\nfirst batch loss: {early:.3f} (predicted: about 1.0)   mean of last 100: {late:.3f}")

## Sampling with the guidance knob

Algorithm 2 from parts 9-10, with one change: at every step the network is called **twice** — once with the target label, once with the null token — and the two calls are combined into $\tilde\epsilon = \epsilon_\varnothing + w\,(\epsilon_y - \epsilon_\varnothing)$ before the update. We target cluster 2 and turn the knob through $w = 0, 1, 3, 7$.

### Step 7 — Sample at w = 0, 1, 3, 7 targeting cluster 2

Predictions to check against: at $w = 0$ the label is ignored, so about 1/8 of samples should land on cluster 2 (the lottery); at $w = 1$ we sample the plain conditional — most samples on target; at $w = 3$ and beyond, nearly all. We assert the on-target fraction rises from $w = 0$ to $w = 3$.

In [ ]:
def sample_cfg(w, target, n_samples):
    yvec = torch.full((n_samples,), target)
    null_vec = torch.full((n_samples,), NULL_LABEL)
    with torch.no_grad():
        xt = torch.randn(n_samples, 2)
        for t in range(T, 0, -1):
            tvec = torch.full((n_samples,), t)
            eps_c = model(xt, tvec, yvec)
            eps_u = model(xt, tvec, null_vec)
            eps_g = eps_u + w * (eps_c - eps_u)
            coef = betas[t - 1] / sqrt_1m_abar[t - 1]
            mean = (xt - coef * eps_g) / torch.sqrt(alphas[t - 1])
            if t > 1:
                z = torch.randn_like(xt)
                xt = mean + torch.sqrt(betas[t - 1]) * z
            else:
                xt = mean
    return xt


angles_8 = 2 * math.pi * torch.arange(8) / 8
centers_8 = 4.0 * torch.stack([torch.cos(angles_8), torch.sin(angles_8)], dim=1)

target = 2
w_list = [0.0, 1.0, 3.0, 7.0]
cfg_samples = {}
frac_on_target = {}

for w in w_list:
    s = sample_cfg(w, target, 400)
    nearest = torch.cdist(s, centers_8).argmin(dim=1)
    cfg_samples[w] = s
    frac_on_target[w] = (nearest == target).float().mean().item()

assert frac_on_target[0.0] < 0.5
assert frac_on_target[3.0] > frac_on_target[0.0]

for w in w_list:
    print(f"w = {w:3.0f}   fraction of samples on cluster {target}: {frac_on_target[w]:.3f}")

### Step 8 — See the knob work

Read the strip left to right. At $w = 0$ the samples cover the whole ring — the label is ignored. At $w = 1$ they gather on cluster 2 with its honest spread. At $w = 3$ and $w = 7$ the pack tightens: obedience keeps rising, but the surviving spread shrinks below the data's true within-cluster spread — the fidelity-diversity trade, in one picture.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, w in zip(axes, w_list):
    pts = cfg_samples[w].numpy()
    ax.scatter(x0_np[:, 0], x0_np[:, 1], s=2, alpha=0.08, color="gray")
    ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.6, color="#ff7b72")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)
    ax.set_title(f"w = {w:g}   ({frac_on_target[w]:.0%} on target)")
    ax.set_xlabel("first coordinate")
axes[0].set_ylabel("second coordinate")
fig.suptitle("Classifier-free guidance targeting cluster 2: turning the knob w")
plt.tight_layout()
plt.show()

spread_w1 = cfg_samples[1.0].std(dim=0).mean().item()
spread_w7 = cfg_samples[7.0].std(dim=0).mean().item()

print(f"sample spread at w = 1: {spread_w1:.3f}   at w = 7: {spread_w7:.3f} — fidelity bought with diversity")

## Latent diffusion in miniature: digits from noise

The course finale. Stable Diffusion runs its whole diffusion loop in an autoencoder's compressed latent space ($512 \times 512 \times 3 = 786{,}432$ pixel numbers down to $64 \times 64 \times 4 = 16{,}384$ latent numbers — a 48-fold shrink), then decodes once at the end. We do the same in miniature: the images are the 8-by-8 digits (64 pixels), the autoencoder is **PCA** — the linear special case, computed exactly by numpy's SVD — and the latent space has 16 dimensions, a 4-fold shrink.

### Step 9 — Compress the digits 64 → 16 with PCA

PCA finds the 16 directions along which the digits vary most. Encoding = center the image and read its 16 coordinates along those directions; decoding = re-add the directions and the mean. Two honesty checks: the 16 directions should keep most of the variance (we assert above 75%), and decoded reconstructions should still be recognizable digits — compression must lose texture, not identity.

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float64)
x_mean = X.mean(axis=0)
Xc = X - x_mean

U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
k = 16
V16 = Vt[:k]

lat = Xc @ V16.T
recon = lat @ V16 + x_mean
explained = float((S[:k] ** 2).sum() / (S ** 2).sum())

assert X.shape == (1797, 64)
assert lat.shape == (1797, 16)
assert explained > 0.75
assert np.abs(recon - X).mean() < 2.0

print(f"digits: {X.shape[0]} images of {X.shape[1]} pixels, values in [{X.min():.0f}, {X.max():.0f}]")
print(f"latents: {lat.shape} — a {X.shape[1] // k}x shrink (Stable Diffusion's is 48x)")
print(f"variance kept by 16 of 64 directions: {explained:.1%}")
print(f"mean absolute reconstruction error: {np.abs(recon - X).mean():.3f} (on a 0-16 pixel scale)")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for j in range(8):
    axes[0, j].imshow(X[j].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[1, j].imshow(np.clip(recon[j], 0, 16).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[0, j].set_xticks([])
    axes[0, j].set_yticks([])
    axes[1, j].set_xticks([])
    axes[1, j].set_yticks([])
axes[0, 0].set_ylabel("original", fontsize=9)
axes[1, 0].set_ylabel("PCA-16", fontsize=9)
fig.suptitle("The miniature autoencoder: 16 numbers keep a digit's identity")
plt.tight_layout()
plt.show()

### Step 10 — Train a DDPM in the 16-number latent space

Encode every digit, standardize each latent coordinate to spread 1 (so the schedule's endpoint $\mathcal{N}(0, \mathbf{I})$ matches the data's scale), and train part 10's unconditional recipe on the latents — same schedule, same loss, same loop, with 16 coordinates in place of 2. Nothing in parts 6-11 cared what the coordinates meant.

In [ ]:
lat_scale = lat.std(axis=0)
lat_std = lat / lat_scale
lat_t = torch.tensor(lat_std, dtype=torch.float32)
n_lat = lat_t.shape[0]

assert abs(float(lat_std.std()) - 1.0) < 0.05


class LatentEpsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(k + emb_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, k),
        )

    def forward(self, x, t):
        temb = time_embedding(t)
        h = torch.cat([x, temb], dim=1)
        return self.net(h)


lat_model = LatentEpsMLP()
opt_lat = torch.optim.Adam(lat_model.parameters(), lr=1e-3)
lat_losses = []

for step in range(1500):
    idx = torch.randint(0, n_lat, (256,))
    z0 = lat_t[idx]
    t = torch.randint(1, T + 1, (256,))
    eps = torch.randn(256, k)
    zt = sqrt_abar[t - 1][:, None] * z0 + sqrt_1m_abar[t - 1][:, None] * eps
    eps_pred = lat_model(zt, t)
    loss = ((eps - eps_pred) ** 2).mean()
    opt_lat.zero_grad()
    loss.backward()
    opt_lat.step()
    lat_losses.append(loss.item())
    if step % 250 == 0 or step == 1499:
        print(f"step {step:4d}   batch loss = {loss.item():.3f}")

assert float(np.mean(lat_losses[-100:])) < lat_losses[0] * 0.7

print(f"\nlatent DDPM trained: first loss {lat_losses[0]:.3f}, last-100 mean {float(np.mean(lat_losses[-100:])):.3f}")

### Step 11 — Generate digits from pure noise (the finale)

Algorithm 2 walks 64 pure-noise latents down all 200 steps, then the PCA decoder inflates each 16-number result back to 64 pixels: un-standardize, re-add the 16 directions, re-add the mean image. Every image below started as Gaussian static — no digit in the grid exists in the dataset.

In [ ]:
with torch.no_grad():
    zt = torch.randn(64, k)
    for t in range(T, 0, -1):
        tvec = torch.full((64,), t)
        eps_pred = lat_model(zt, tvec)
        coef = betas[t - 1] / sqrt_1m_abar[t - 1]
        mean = (zt - coef * eps_pred) / torch.sqrt(alphas[t - 1])
        if t > 1:
            zkick = torch.randn_like(zt)
            zt = mean + torch.sqrt(betas[t - 1]) * zkick
        else:
            zt = mean

gen_lat = zt.numpy() * lat_scale
gen_imgs = gen_lat @ V16 + x_mean
gen_grid = gen_imgs.reshape(64, 8, 8)

fig, axes = plt.subplots(8, 8, figsize=(6.5, 6.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(gen_grid[i], 0, 16), cmap="gray_r", vmin=0, vmax=16)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("64 digit images generated from pure noise (latent DDPM + PCA decoder)")
plt.tight_layout()
plt.show()

print("every image above began as 16 numbers of Gaussian static")

### Step 12 — Score the generated digits

Eyes first, then numbers. The checks: generated pixel values should stay in a sane range before clipping — within one full pixel-scale (16) beyond either end of the real 0-16 range, since the PCA decoder overshoots a little even on real digits — and the generated batch's per-pixel spread should be within a factor of 2 of the real digits'. Together: the samples live in the same statistical neighborhood as the data.

In [ ]:
real_pp_std = X.std(axis=0)
gen_pp_std = gen_imgs.std(axis=0)
std_ratio = float(gen_pp_std.mean() / real_pp_std.mean())

assert gen_imgs.min() > -16.0
assert gen_imgs.max() < 32.0
assert 0.5 < std_ratio < 2.0

print(f"generated pixel range before clipping: [{gen_imgs.min():.1f}, {gen_imgs.max():.1f}] (real: [0, 16])")
print(f"per-pixel spread, generated vs real: {gen_pp_std.mean():.2f} vs {real_pp_std.mean():.2f} (ratio {std_ratio:.2f})")
print("within a factor of 2 — the generated batch matches the data's statistics, and your eyes confirm the rest")

## Practice

The lesson's six problems. The arithmetic ones are checked in code; the reasoning ones have full model answers. Try each in the empty cell, then reveal the worked solution.

**Problem 1.** CFG arithmetic. At some sampling step the two calls return $\epsilon_\varnothing = (0.4,\ 0.0)$ and $\epsilon_y = (0.1,\ 0.6)$. Compute $\tilde\epsilon$ for $w = 0, 1, 2, 5$ by hand, then check in code. What pattern do the four answers make?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
eps_u2 = torch.tensor([0.4, 0.0])
eps_c2 = torch.tensor([0.1, 0.6])
direction2 = eps_c2 - eps_u2

for w in [0.0, 1.0, 2.0, 5.0]:
    guided = eps_u2 + w * direction2
    print(f"w = {w:g}   eps_tilde = {guided.tolist()}")
```

By hand: the steering direction is $(0.1 - 0.4,\ 0.6 - 0.0) = (-0.3,\ 0.6)$, reused for every $w$.

- $w = 0$: $(0.4,\ 0.0)$ — the unconditional guess, label ignored.
- $w = 1$: $(0.1,\ 0.6)$ — exactly $\epsilon_y$, plain conditional generation.
- $w = 2$: $(0.4 - 0.6,\ 0.0 + 1.2) = (-0.2,\ 1.2)$.
- $w = 5$: $(0.4 - 1.5,\ 0.0 + 3.0) = (-1.1,\ 3.0)$.

**Answer:** all four sit on the straight line through $\epsilon_\varnothing$ and $\epsilon_y$, with $w$ the position along it — at the base for $w = 0$, on the conditional guess at $w = 1$, extrapolated past it for $w > 1$. Guidance never bends; it slides along the label's direction.

</details>

**Problem 2.** Derive the CFG formula. Starting from the sharpened target density proportional to $q(x_t)\, q(y \mid x_t)^w$, derive $\tilde\epsilon = \epsilon_\varnothing + w\,(\epsilon_y - \epsilon_\varnothing)$, citing the rule behind each move.

In [ ]:
# Your turn (write the derivation as comments, or verify a step numerically):


<details><summary>Show worked solution</summary>

1. **Log of the target:** $\log q(x_t) + w \log q(y \mid x_t) + \text{const}$ — log of a product is a sum of logs, log of a power is the power times the log (part 5); the normalizer is constant in $x_t$.
2. **Gradient in $x_t$:** $\nabla \log q(x_t) + w\, \nabla \log q(y \mid x_t)$ — slopes add, a constant's slope is zero, scaling by $w$ scales the slope (part 11's slope facts).
3. **Rearrange Bayes on scores:** from $\nabla \log q(x_t \mid y) = \nabla \log q(x_t) + \nabla \log q(y \mid x_t)$, subtract $\nabla \log q(x_t)$ from both sides: $\nabla \log q(y \mid x_t) = \nabla \log q(x_t \mid y) - \nabla \log q(x_t)$.
4. **Substitute:** guided score $= \nabla \log q(x_t) + w\,\big(\nabla \log q(x_t \mid y) - \nabla \log q(x_t)\big)$.
5. **Convert scores to noise guesses:** multiply the whole equation by $-\sqrt{1-\bar\alpha_t}$ and apply part 11's bridge $\nabla \log q = -\epsilon^{*}/\sqrt{1-\bar\alpha_t}$ to each score — the unconditional score becomes $\epsilon_\varnothing$, the conditional one becomes $\epsilon_y$ (part 11's derivation reruns word for word on the label-$y$ sub-population).

**Answer:** $\tilde\epsilon = \epsilon_\varnothing + w\,(\epsilon_y - \epsilon_\varnothing)$ — two forward calls and a knob, no classifier anywhere.

</details>

**Problem 3.** Classifier versus classifier-free. Your team has a large, frozen, unconditional diffusion model. Colleague A proposes classifier guidance; colleague B proposes classifier-free guidance. What does each require, what does each cost per sampling step, and which do you pick if retraining the diffusion model is off the table?

In [ ]:
# Your turn (reason in comments):


<details><summary>Show worked solution</summary>

- **Plan A (classifier guidance)** requires training a *new, noise-robust* classifier $q(y \mid x_t)$ — accurate at every noise level $t$, so it must be trained on noised inputs — while the diffusion model stays frozen. Per sampling step: one diffusion call plus a classifier forward-and-backward pass (the tilt is a gradient). Steering quality is capped by the classifier: wherever it is confidently wrong on noisy inputs, the tilt points confidently the wrong way.
- **Plan B (classifier-free guidance)** requires the diffusion network itself to accept the label — trained with label dropout so one set of weights learns both the conditional and unconditional guess. A frozen unconditional model has no label input, so plan B needs retraining or fine-tuning. Per sampling step: two forward passes of the one network, no gradients, no second model, no classifier ceiling.

**Answer:** with retraining off the table, only plan A is available — bolt a noise-level classifier onto the frozen model. When the constraint lifts, plan B is simpler and stronger (one model, forward passes only), which is why it is the modern default.

</details>

**Problem 4.** The cost argument at other sizes. (a) Count the dimensions of a 1024-by-1024 color image diffused in pixel space. (b) Count its 128-by-128-by-4 latent and the shrink factor. (c) Repeat for 256-by-256-by-3 with a 32-by-32-by-4 latent. Check all three in code.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

```python
pix_1024 = 1024 * 1024 * 3
lat_128 = 128 * 128 * 4
pix_256 = 256 * 256 * 3
lat_32 = 32 * 32 * 4

print(f"1024x1024x3 image: {pix_1024:,} dims   128x128x4 latent: {lat_128:,} dims   ratio {pix_1024 // lat_128}")
print(f"256x256x3 image:   {pix_256:,} dims    32x32x4 latent:   {lat_32:,} dims    ratio {pix_256 // lat_32}")

assert pix_1024 // lat_128 == 48
assert pix_256 // lat_32 == 48
```

**Answer:** (a) $1024 \times 1024 \times 3 = 3{,}145{,}728$. (b) $128 \times 128 \times 4 = 65{,}536$ — a 48-fold shrink. (c) $196{,}608$ versus $4{,}096$ — 48 again. The ratio repeats because the geometry repeats: each grid side shrinks 8-fold ($8 \times 8 = 64$) while channels grow from 3 to 4, and $64 \times 3/4 = 48$. Every tensor in the diffusion loop shrinks by that factor, at every sampler step and every training step.

</details>

**Problem 5.** Map the machine. For each Stable Diffusion component, name the course part where you learned it: (a) the noise schedule; (b) the U-Net noise-guesser with its time embedding; (c) the training loss; (d) the 30-step deterministic sampler; (e) the guidance slider $w$; (f) the encoder/decoder around the loop; (g) the text encoder.

In [ ]:
# Your turn (write your mapping as comments):


<details><summary>Show worked solution</summary>

| Component | Where you learned it |
|---|---|
| (a) noise schedule $\beta_t$, $\bar\alpha_t$, one-line sampler | part 6 (chains: part 4) |
| (b) U-Net noise-guesser + sinusoidal time embedding | part 10 (our MLP is the small honest version) |
| (c) training loss: ELBO $\to$ KL terms $\to$ $L_{\mathrm{simple}}$ | parts 8-9 |
| (d) fast deterministic sampler (scores, Tweedie, $\hat{x}_0$, DDIM) | part 11 |
| (e) guidance: two calls and the knob $w$ | part 12 |
| (f) autoencoder: encode 48-fold, diffuse in latents, decode once | part 12 (our PCA miniature; background: the VAE lesson) |
| (g) text encoder (CLIP) | interface = part 12's embedding contract; internals imported from the transformers world |

**Answer:** every box except the text encoder's internals maps to a derivation you did by hand — which was the course's promise in part 1.

</details>

**Problem 6.** Design: two conditions at once. You want digits conditioned on TWO labels — digit identity (0-9) and stroke thickness (thin/thick) — steerable independently at sampling time. Design the training recipe and sampling update, and name the assumption the update leans on.

In [ ]:
# Your turn (sketch the design in comments):


<details><summary>Show worked solution</summary>

- **Architecture:** one embedding table per condition, each with its own null token — 10+1 rows for digit, 2+1 for thickness — both embeddings handed to the network alongside the time embedding: $\epsilon_\theta(x_t, t, y_1, y_2)$.
- **Training:** independent dropout — replace $y_1$ by $\varnothing_1$ with probability 0.1 and, independently, $y_2$ by $\varnothing_2$ with probability 0.1 — so the network trains on all four patterns: both labels, either alone, neither.
- **Sampling:** sharpen each condition separately (target $\propto q(x_t)\, q(y_1 \mid x_t)^{w_1}\, q(y_2 \mid x_t)^{w_2}$) and rerun part 12's derivation term by term:

$$\tilde\epsilon = \epsilon_{\varnothing} + w_1\,(\epsilon_{y_1} - \epsilon_{\varnothing}) + w_2\,(\epsilon_{y_2} - \epsilon_{\varnothing})$$

three forward calls per step ($\epsilon_{y_1}$ has the digit label with thickness nulled, $\epsilon_{y_2}$ the reverse), one knob per condition.

- **The assumption:** summing the two tilts treats the conditions as steering independently — a factorization bet on $q(y_1, y_2 \mid x_t)$. For strongly interacting conditions (a rare combination like "digit 1, very thick"), the summed tilt can point where neither condition endorses; the honest fix is to also train and query the fully joint call $\epsilon_\theta(x_t, t, y_1, y_2)$.

**Answer:** two tables + independent dropout + the three-call update above; it leans on an independence assumption between the conditions, which part 2 taught us to state rather than assume silently.

</details>

## Wrap-up

The course is complete, and this notebook verified its last claims: the CFG arithmetic $(2.0, 1.1)$ to the decimal, its $w = 0$ and $w = 1$ endpoints, a label-dropout-trained conditional model whose on-target fraction climbs from the 1/8 lottery at $w = 0$ toward 1 as the knob turns (with the spread shrinking as the price), PCA keeping most of the digits' variance in a 4-fold-smaller latent space, and a latent DDPM whose decoded samples are recognizable digits with per-pixel statistics within a factor of 2 of the real data — images generated from pure noise, on a CPU, by machinery you derived one operation at a time. From here: the paper lessons on DDPM, classifier-free guidance, and latent diffusion read the original sources with the eyes you now have, and the diffusion capstone puts the full pipeline in your hands at real-image scale.